# Employee Salary Visualization

This notebook loads employee data from the Neon PostgreSQL database and compares average salary by position and employee start year.

## Load Employee Data

The complete `employees` table is queried with `psycopg2` and loaded into a Pandas DataFrame.

In [1]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / '.env')
DATABASE_URL = os.getenv('DATABASE_URL')

with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute('SELECT * FROM employees')
        rows = cursor.fetchall()
        columns = [column[0] for column in cursor.description]

df = pd.DataFrame(rows, columns=columns)
df['start_date'] = pd.to_datetime(df['start_date'])
df['start_year'] = df['start_date'].dt.year
df.head()

,employee_id,name,start_date,salary,department_id,start_year
0,60,Elizabeth Peterson,2019-01-15,169868,1,2019
1,68,Amanda Weaver,2019-12-11,135953,10,2019
2,76,Tina Tanner,2019-12-07,109270,2,2019
3,77,Brian Perry,2015-12-29,159882,5,2015
4,80,Craig Maldonado,2021-07-26,61523,2,2021


## Join Department Information

The Neon `departments` table uses `department_id` as its primary key. Each employee receives a random department ID, stored as a foreign key in `employees`. The redundant `position` column is removed from `employees`, and department details are retrieved through the join.

In [2]:
department_rows = [
    (1, 'Finance', 'New York', 850000),
    (2, 'Human Resources', 'Chicago', 700000),
    (3, 'Marketing', 'Austin', 950000),
    (4, 'Sales', 'Boston', 1100000),
    (5, 'Finance', 'New York', 850000),
    (6, 'Operations', 'Dallas', 1250000),
    (7, 'Creative Services', 'Austin', 600000),
    (8, 'Customer Experience', 'Denver', 900000),
    (9, 'Operations', 'Dallas', 1000000),
    (10, 'Administration', 'Chicago', 500000),
]

with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute('ALTER TABLE employees ADD COLUMN IF NOT EXISTS department_id INTEGER')
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS departments (
                department_id INTEGER PRIMARY KEY,
                department TEXT NOT NULL,
                location TEXT NOT NULL,
                annual_budget INTEGER NOT NULL CHECK (annual_budget > 0)
            )
            """
        )
        cursor.executemany(
            """
            INSERT INTO departments (department_id, department, location, annual_budget)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (department_id) DO UPDATE SET
                department = EXCLUDED.department,
                location = EXCLUDED.location,
                annual_budget = EXCLUDED.annual_budget
            """,
            department_rows,
        )
        cursor.execute(
            """
            UPDATE employees
            SET department_id = FLOOR(random() * 10)::integer + 1
            """
        )
        cursor.execute(
            """
            ALTER TABLE employees
            DROP CONSTRAINT IF EXISTS employees_department_id_fkey
            """
        )
        cursor.execute(
            """
            ALTER TABLE employees
            ADD CONSTRAINT employees_department_id_fkey
            FOREIGN KEY (department_id) REFERENCES departments(department_id)
            """
        )
        cursor.execute('ALTER TABLE employees DROP COLUMN IF EXISTS position')
        cursor.execute(
            """
            SELECT
                e.employee_id,
                e.name,
                e.start_date,
                e.salary,
                EXTRACT(YEAR FROM e.start_date)::integer AS start_year,
                e.department_id,
                d.department,
                d.location,
                d.annual_budget
            FROM employees AS e
            INNER JOIN departments AS d
                ON e.department_id = d.department_id
            """
        )
        joined_rows = cursor.fetchall()
        joined_columns = [column[0] for column in cursor.description]

employee_department_df = pd.DataFrame(joined_rows, columns=joined_columns)
employee_department_df.head()

,employee_id,name,start_date,salary,start_year,department_id,department,location,annual_budget
0,60,Elizabeth Peterson,2019-01-15,169868,2019,3,Marketing,Austin,950000
1,68,Amanda Weaver,2019-12-11,135953,2019,3,Marketing,Austin,950000
2,76,Tina Tanner,2019-12-07,109270,2019,9,Operations,Dallas,1000000
3,77,Brian Perry,2015-12-29,159882,2015,1,Finance,New York,850000
4,80,Craig Maldonado,2021-07-26,61523,2021,1,Finance,New York,850000


## Average Salary by Position and Start Year

The grouped bar chart uses one bar for each start year within every position.

In [3]:
average_salary = (
    df.groupby(['position', 'start_year'], as_index=False)['salary']
    .mean()
    .pivot(index='position', columns='start_year', values='salary')
)

ax = average_salary.plot(kind='bar', figsize=(15, 8), width=0.85)
ax.set_title('Average Salary by Position and Start Year')
ax.set_xlabel('Position')
ax.set_ylabel('Average Salary')
ax.legend(title='Start Year', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

KeyError: 'position'

## Project Assignments and Distribution Analysis

Each employee is assigned to a project in Neon. The joined project dataset is used to compare salary and years-of-service distributions by project and department.

In [ ]:
import random

project_rows = [
    ('Project Atlas', 'Operations'),
    ('Project Beacon', 'Finance'),
    ('Project Cedar', 'Marketing'),
    ('Project Delta', 'Sales'),
    ('Project Ember', 'Human Resources'),
]

with psycopg2.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS project_assignments (
                assignment_id BIGSERIAL PRIMARY KEY,
                employee_id INTEGER NOT NULL UNIQUE REFERENCES employees(employee_id),
                project_name TEXT NOT NULL,
                project_department TEXT NOT NULL
            )
            """
        )
        cursor.execute('SELECT employee_id FROM employees ORDER BY employee_id')
        employee_ids = [row[0] for row in cursor.fetchall()]
        assignments = [
            (employee_id, *random.choice(project_rows))
            for employee_id in employee_ids
        ]
        cursor.executemany(
            """
            INSERT INTO project_assignments (employee_id, project_name, project_department)
            VALUES (%s, %s, %s)
            ON CONFLICT (employee_id) DO UPDATE SET
                project_name = EXCLUDED.project_name,
                project_department = EXCLUDED.project_department
            """,
            assignments,
        )
        cursor.execute(
            """
            SELECT
                e.employee_id,
                e.name,
                e.salary,
                e.start_date,
                d.department_id,
                d.department,
                pa.project_name,
                pa.project_department
            FROM employees AS e
            INNER JOIN departments AS d
                ON e.department_id = d.department_id
            INNER JOIN project_assignments AS pa
                ON e.employee_id = pa.employee_id
            """
        )
        project_rows_from_db = cursor.fetchall()
        project_columns = [column[0] for column in cursor.description]

project_df = pd.DataFrame(project_rows_from_db, columns=project_columns)
project_df['start_date'] = pd.to_datetime(project_df['start_date'])
project_df['years_of_service'] = (
    (pd.Timestamp.today().normalize() - project_df['start_date']).dt.days / 365.25
).clip(lower=0).round(1)

print('Project assignment rows:', len(project_df))
display(project_df.head())

salary_distribution = project_df.groupby(
    ['project_name', 'department'],
)['salary'].describe().round(2)
years_distribution = project_df.groupby(
    ['project_name', 'department'],
)['years_of_service'].describe().round(2)

print('Salary distribution by project and department:')
display(salary_distribution)
print('Years-of-service distribution by project and department:')
display(years_distribution)

## Section 1: Department Salary Heatmap

This heatmap shows the average salary for each department across start years 2019 through 2024. Darker cells represent higher average salaries.

### Main Findings in the Graph:

- Highest Peak Salaries: Marketing reached the top single-cohort average ($173,101 in 2020), followed by Administration ($166,772 in 2023) and Finance ($165,228 in 2019).   
- Lowest Pay Averages: The lowest starting averages were recorded in Human Resources ($60,000 in 2024), Customer Experience ($65,226 in 2024), and Creative Services ($66,635 in 2022).   
- Hiring Gaps: Blank white cells highlight years with no hires in specific departments (e.g., Marketing in 2021; Administration in 2020 & 2024).   
- Pay Trend over Time: Earlier hire cohorts (2019–2021) generally average higher salaries compared to 2024 hires.  

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

analysis_df = project_df.copy()
analysis_df['start_year'] = analysis_df['start_date'].dt.year
analysis_df = analysis_df[analysis_df['start_year'].between(2019, 2024)]

salary_heatmap = analysis_df.pivot_table(
    index='department',
    columns='start_year',
    values='salary',
    aggfunc='mean',
)

fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(salary_heatmap, aspect='auto', cmap='YlGnBu')
ax.set_title('Average Salary by Department and Start Year')
ax.set_xlabel('Start Year')
ax.set_ylabel('Department')
ax.set_xticks(range(len(salary_heatmap.columns)), salary_heatmap.columns)
ax.set_yticks(range(len(salary_heatmap.index)), salary_heatmap.index)
for row_index in range(salary_heatmap.shape[0]):
    for column_index in range(salary_heatmap.shape[1]):
        value = salary_heatmap.iloc[row_index, column_index]
        if not np.isnan(value):
            ax.text(column_index, row_index, f'${value:,.0f}', ha='center', va='center')
fig.colorbar(image, ax=ax, label='Average Salary')
fig.tight_layout()
plt.show()

## Average Salary by Department

Each point represents one department's average salary and average years of service for employees who started between 2019 and 2024.

### Main Findings in the Graph:

- Top Earning Departments: Administration ranks highest in average salary (> $130,000), followed by Sales (~$126,400) and Operations (~$124,400).   
- Key Departmental Outliers:
    - Human Resources Highest average work time (~5.35 years) but lowest average salary (~$103,700).   
    - Creative Services: High average work time (~5.35 years) with mid-tier pay (~$118,300).   
    - Marketing: Lowest average work time (~3.82 years) with a mid-range average salary (~$112,200).   

In [ ]:
department_salary_summary = (
    analysis_df.groupby('department', as_index=False)
    .agg(
        mean_salary=('salary', 'mean'),
        mean_years_of_service=('years_of_service', 'mean'),
    )
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    department_salary_summary['mean_years_of_service'],
    department_salary_summary['mean_salary'],
    s=100,
    alpha=0.85,
)

for _, department_row in department_salary_summary.iterrows():
    ax.annotate(
        department_row['department'],
        (
            department_row['mean_years_of_service'],
            department_row['mean_salary'],
        ),
        xytext=(6, 6),
        textcoords='offset points',
    )

trend_x = department_salary_summary['mean_years_of_service'].to_numpy()
trend_y = department_salary_summary['mean_salary'].to_numpy()
trend_slope, trend_intercept = np.polyfit(trend_x, trend_y, 1)
line_x = np.linspace(trend_x.min(), trend_x.max(), 100)
ax.plot(
    line_x,
    trend_slope * line_x + trend_intercept,
    color='black',
    linestyle='--',
    label='Department-average trendline',
)
ax.set_title('Average Salary by Department (2019-2024)')
ax.set_xlabel('Average Years of Service')
ax.set_ylabel('Average Salary')
ax.legend()
fig.tight_layout()
plt.show()

## Section 3: Department Employee Service Trends

This line chart shows the average years of service for employees in each department by their start year, using the 2019-2024 analysis range.

### Main Findings in the Graph:

- Down-salary Trend Across All Departments: 
    - Average years of service downgraded similar linearly for every department as the start year advances from 2019 (~6.8–7.5 years) down to 2024 (~1.8–2.6 years).   
    - Highest work time Cohort (2019): Finance recorded the highest average work time in the 2019 cohort (~7.4 years), closely followed by Operations (~7.3 years).   
    - Lowest Tenure Cohort (2024): Customer Experience dropped to the lowest average work time among 2024 hires (~1.8 years), while Sales retained the highest relative work time in that same cohort (~2.6 years).   
    - Consistent Progression: Department lines track closely together throughout the timeline, reflecting a standard mathematical reduction in work time inherent to more recent start dates.   

In [ ]:
yearly_department_service = (
    analysis_df.groupby(['department', 'start_year'], as_index=False)['years_of_service']
    .mean()
)

fig, ax = plt.subplots(figsize=(14, 8))
for department, department_data in yearly_department_service.groupby('department'):
    ax.plot(
        department_data['start_year'],
        department_data['years_of_service'],
        marker='o',
        linewidth=2,
        label=department,
    )

ax.set_title('Department Employee Years-of-Service Trend (2019-2024)')
ax.set_xlabel('Employee Start Year')
ax.set_ylabel('Average Years of Service')
ax.set_xticks(range(2018, 2025))
ax.legend(title='Department', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()